# Demo: Jaynes-Cummings model scan

This notebook builds photon-number probability grids from the analytical
Jaynes-Cummings model, then classifies each grid point with the numerical LP
solver and the analytical criteria. Unlike the probability-plane demo, the two
scan coordinates here are physical model parameters: $\tau$ and $\nu$.

Non-physical probability vectors, such as negative values or selected probabilities with sum larger than one, are skipped by the scan, stored as `NaN`, and drawn in white in the maps.


In [ ]:
# Make the project importable when running from the notebooks/ directory.
import os, sys
sys.path.insert(0, os.path.abspath('..'))

from src.config import apply_thread_limits
apply_thread_limits()

import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from src.analytical.jaynes_cummings import Pjaycum
from src.comparison import analyze_2d

## Helpers

In [ ]:
def make_jc_grid(tau, nu, pe, indx):
    """Build Pall[i_nu, i_tau, j] = P_{indx[j]}(nu[i_nu], tau[i_tau])."""
    xtau = tau[None, :]
    ynu = nu[:, None]
    return np.stack(
        [Pjaycum(n=int(n), nu=ynu, tau=xtau, pe=pe) for n in indx],
        axis=-1,
    )


def load_scan_archive(path):
    """Read an analyze_2d .npy archive back into its components."""
    with open(path, 'rb') as f:
        x_values = np.load(f)
        y_values = np.load(f)
        fixed_values = np.load(f)
        boundaries = np.load(f, allow_pickle=True)
        boundaries_ana = np.load(f, allow_pickle=True)
        boundaries_anaK = np.load(f, allow_pickle=True)
        boundaries_anaM = np.load(f, allow_pickle=True)
        nonclass_num = np.load(f)
        nonclass_ana = np.load(f)
    return dict(x=x_values, y=y_values, fixed=fixed_values,
                boundaries=boundaries, boundaries_ana=boundaries_ana,
                boundaries_anaK=boundaries_anaK, boundaries_anaM=boundaries_anaM,
                nonclass_num=nonclass_num, nonclass_ana=nonclass_ana)


def draw_boundary_curves(boundaries, *, color, label, mode='scatter', lw=2, s=4, min_points=5, max_jump=0.15):
    first = True
    for b in boundaries:
        if len(b) < min_points:
            continue

        if mode == 'scatter':
            plt.scatter(b[:, 0], b[:, 1], s=s, color=color, label=label if first else None)
        elif mode == 'line':
            plt.plot(b[:, 0], b[:, 1], lw=lw, color=color, label=label if first else None)
        elif mode == 'cut-line':
            jumps = np.sqrt(np.sum(np.diff(b, axis=0)**2, axis=1))
            cuts = np.where(jumps > max_jump)[0] + 1
            for piece in np.split(b, cuts):
                if len(piece) < min_points:
                    continue
                plt.plot(piece[:, 0], piece[:, 1], lw=lw, color=color, label=label if first else None)
                first = False
            continue
        else:
            raise ValueError("mode must be 'scatter', 'line', or 'cut-line'")

        first = False


def plot_boundaries(boundaries, boundaries_anaK, boundaries_anaM, title, mode='scatter'):
    plt.figure(figsize=(6, 5))
    draw_boundary_curves(boundaries, color='black', label='LP', mode=mode, lw=4, s=5)
    draw_boundary_curves(boundaries_anaK, color='red', label='analytical K', mode=mode, lw=3, s=4)
    draw_boundary_curves(boundaries_anaM, color='blue', label='analytical M', mode=mode, lw=3, s=4)

    handles, labels = plt.gca().get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    if by_label:
        plt.legend(by_label.values(), by_label.keys())
    plt.xlim(0, float(np.max(tau)))
    plt.ylim(0, float(np.max(nu)))
    plt.xlabel(r'$\tau$')
    plt.ylabel(r'$\nu$')
    plt.title(title)
    plt.tight_layout()
    plt.show()


def classification_map(data):
    """0 classical, 1 analytical-only, 2 LP-only, 3 both, NaN non-physical."""
    num = data['nonclass_num']
    ana = data['nonclass_ana']
    valid = np.isfinite(num) & np.isfinite(ana)
    classes = np.full(num.shape, np.nan, dtype=float)
    classes[valid] = 0.0
    classes[valid & (ana > 0.5)] = 1.0
    classes[valid & (num > 0.5)] = 2.0
    classes[valid & (num > 0.5) & (ana > 0.5)] = 3.0
    return classes


def map_cmap():
    cmap = plt.get_cmap('viridis').copy()
    cmap.set_bad('white')
    return cmap


def plot_map(data, title, boundary_mode='scatter'):
    classes = classification_map(data)
    plt.figure(figsize=(6, 5))
    plt.pcolormesh(
        data['x'], data['y'],
        np.ma.masked_invalid(classes),
        shading='auto', cmap=map_cmap(),
    )
    plt.colorbar(label='0 classical / 1 analytical-only / 2 LP-only / 3 both')
    draw_boundary_curves(data['boundaries'], color='red', label='LP boundary', mode=boundary_mode, lw=2, s=3)
    plt.xlim(0, float(np.max(data['x'])))
    plt.ylim(0, float(np.max(data['y'])))
    plt.xlabel(r'$\tau$')
    plt.ylabel(r'$\nu$')
    plt.title(title)
    plt.tight_layout()
    plt.show()


def run_case(case, tau, nu):
    indx = np.asarray(case['indx'], dtype=int)
    Pall = make_jc_grid(tau=tau, nu=nu, pe=case['pe'], indx=indx)
    os.makedirs(os.path.dirname(os.path.abspath(case['fi'])), exist_ok=True)

    result = analyze_2d(
        Pall,
        x_values=tau,
        y_values=nu,
        indx=indx,
        fi=case['fi'],
        fixed_values=np.array([case['pe']]),
        use_analytical_m=case['use_analytical_m'],
    )
    boundaries, boundaries_ana, boundaries_anaK, boundaries_anaM = result
    print(f"saved: {case['fi']}")
    print(f"grid: {Pall.shape[:2]} | indices: {indx.tolist()}")
    print(f"LP curves: {len(boundaries)} | K curves: {len(boundaries_anaK)} | M curves: {len(boundaries_anaM)}")
    return result

## Grid

The production grid from the scripts can use `step = 0.02`. The default below
is intentionally coarser so the notebook can be executed interactively first.

In [ ]:
stepx = 0.1
stepy = 0.06
tau = np.arange(1e-4, 10.0001, stepx)
nu = np.arange(1e-4, 6.0001,stepy)

print('N_tau =', tau.shape[0], '| N_nu =', nu.shape[0], '| grid points =', tau.shape[0] * nu.shape[0])

The selected-index analytical-M case is disabled by default. For
`indx=[2,4,6,8,10]` the majorization pairs are not covered by the consecutive
precomputed archives, so they must be generated on the fly and can be very
slow.

In [ ]:
try_analytical_m_for_selected = False

## Example 1: $N_P=6$, $p_e=0.3$, all indices

In [ ]:
case_1 = dict(
    pe=0.3,
    indx=np.arange(6),
    fi='../results/demo_3_jc_NP6_pe03.npy',
    use_analytical_m=True,
)

boundaries, boundaries_ana, boundaries_anaK, boundaries_anaM = run_case(case_1, tau, nu)
plot_boundaries(boundaries, boundaries_anaK, boundaries_anaM, r'JC boundaries: $N_P=6$, $p_e=0.3$')

data = load_scan_archive(case_1['fi'])
plot_map(data, r'JC map: $N_P=6$, $p_e=0.3$')

## Example 2: $N_P=12$, $p_e=0.4$, all indices, no analytical-M

In [ ]:
case_2 = dict(
    pe=0.4,
    indx=np.arange(12),
    fi='../results/demo_3_jc_NP12_pe04.npy',
    use_analytical_m=False,
)

boundaries, boundaries_ana, boundaries_anaK, boundaries_anaM = run_case(case_2, tau, nu)
plot_boundaries(boundaries, boundaries_anaK, boundaries_anaM, r'JC boundaries: $N_P=12$, $p_e=0.4$')

data = load_scan_archive(case_2['fi'])
plot_map(data, r'JC map: $N_P=12$, $p_e=0.4$')

## Example 3: selected indices $[2,4,6,8]$, $p_e=0.4$

In [ ]:
case_3 = dict(
    pe=0.4,
    indx=np.array([2, 4, 6, 7, 8]),
    fi='../results/demo_3_jc_selected_2_4_6_7_8_pe04.npy',
    use_analytical_m=True,
)

boundaries, boundaries_ana, boundaries_anaK, boundaries_anaM = run_case(case_3, tau, nu)
plot_boundaries(boundaries, boundaries_anaK, boundaries_anaM, r'JC boundaries: selected indices, $p_e=0.4$')

data = load_scan_archive(case_3['fi'])
plot_map(data, r'JC map: selected indices, $p_e=0.4$')